# Bi-Temporal Change Detection
**NB 04** | Kyiv, Ukraine 2021→2024 | ChangeFormer

In [1]:
import warnings; warnings.filterwarnings('ignore')
import json
import pathlib

import pygeovision as pgv

client = pgv.PyGeoVision()
print(client)

from pygeovision.models.change_detection.changeformer import ChangeDetection


10:29:52 INFO [      engine] PyGeoFetch ready


2026-07-31 10:29:52,339 [INFO] pygeofetch.core.engine: PyGeoFetch ready


10:29:52 INFO [      engine] PyGeoFetch ready


2026-07-31 10:29:52,343 [INFO] pygeofetch.core.engine: PyGeoFetch ready
2026-07-31 10:29:52,345 [INFO] pygeovision.data.pgf_bridge: PyGeoFetch v1.0 Python API active — using native search/download
2026-07-31 10:29:52,346 [INFO] pygeovision.data.pgf_bridge: PyGeoFetch v2.0 processing engine detected — using native preprocessing
2026-07-31 10:29:54,197 [INFO] pygeovision.models.registry: Model registry: 121 architectures loaded


PyGeoVision(v2.1.6 | pygeofetch=✓ | pgf_v2=✓v2 | ai=✓torch | datasets=503 | models=98 | pipelines=51 | validator=✓ | preprocess=✓ | indices=22 | postprocess=✓ | sar=✓ | labeling=7src | losses=10 | inference=4 | xai=4 | monitoring=3 | cloud=3·aws·azure·gcp | edge=ONNX+Jetson | vlm=CLIP+Moon | few_shot | timeseries | 3D)


In [3]:
BBOX        = (30.48, 50.40, 30.58, 50.50)
DATE_BEFORE = ('2021-04-01', '2021-06-30')
DATE_AFTER  = ('2024-04-01', '2024-06-30')
PROVIDERS   = ['planetary_computer']
BANDS       = ['B02','B03','B04','B08']
DATA_DIR    = pathlib.Path('./results/nb04')
DATA_DIR.mkdir(parents=True, exist_ok=True)

pre_results  = client.search(bbox=BBOX, date_range=DATE_BEFORE, providers=PROVIDERS, cloud_cover_max=10)
post_results = client.search(bbox=BBOX, date_range=DATE_AFTER,  providers=PROVIDERS, cloud_cover_max=10)
print(f"Pre-event : {len(pre_results)} scenes | Post-event: {len(post_results)} scenes")


Pre-event : 20 scenes | Post-event: 27 scenes


In [4]:
PRE_DIR  = DATA_DIR / 'pre';  PRE_DIR.mkdir(exist_ok=True)
POST_DIR = DATA_DIR / 'post'; POST_DIR.mkdir(exist_ok=True)

pre_dl  = client.download(pre_results[:1],  output_dir=str(PRE_DIR),bands=BANDS, post_process=['reproject:EPSG:32637','cog'])
post_dl = client.download(post_results[:1], output_dir=str(POST_DIR),bands=BANDS, post_process=['reproject:EPSG:32637','cog'])

pre_scene  = pre_dl[0].path  if pre_dl  and pre_dl[0].success  else None
post_scene = post_dl[0].path if post_dl and post_dl[0].success else None
print(f"Pre : {pre_scene}\nPost: {post_scene}")


2026-07-31 10:30:11,006 [INFO] pygeovision.data.fetch: Downloading 1 pystac-fallback item(s) directly (bypassing PyGeoFetch engine).
2026-07-31 10:30:13,255 [INFO] pygeovision.data.fetch:   [direct] S2A_MSIL2A_20210621T085601_R007_T36UUB_20210624T122020  B02 → S2A_MSIL2A_20210621T085601_R007_T36UUB_20210624T122020_B02.tif
2026-07-31 10:31:16,575 [INFO] pygeovision.data.fetch:   [direct] S2A_MSIL2A_20210621T085601_R007_T36UUB_20210624T122020  B03 → S2A_MSIL2A_20210621T085601_R007_T36UUB_20210624T122020_B03.tif
2026-07-31 10:32:28,527 [INFO] pygeovision.data.fetch:   [direct] S2A_MSIL2A_20210621T085601_R007_T36UUB_20210624T122020  B04 → S2A_MSIL2A_20210621T085601_R007_T36UUB_20210624T122020_B04.tif
2026-07-31 10:33:18,892 [INFO] pygeovision.data.fetch:   [direct] S2A_MSIL2A_20210621T085601_R007_T36UUB_20210624T122020  B08 → S2A_MSIL2A_20210621T085601_R007_T36UUB_20210624T122020_B08.tif
2026-07-31 10:34:42,823 [INFO] pygeovision.data.fetch:   ✓ S2A_MSIL2A_20210621T085601_R007_T36UUB_20210

Pre : results/nb04/pre/planetary_computer/S2A_MSIL2A_20210621T085601_R007_T36UUB_20210624T122020_repr_cog.tif
Post: results/nb04/post/planetary_computer/S2A_MSIL2A_20240628T090601_R050_T36UUA_20240628T131515_repr_cog.tif


In [5]:
# Validate both scenes before preprocessing
for label, path in [('PRE', pre_scene), ('POST', post_scene)]:
    if path:
        rep = client.validator.validate(str(path), required_bands=len(BANDS))
        print(f"{label}: {'PASSED' if rep.passed else 'ISSUES'} | {rep.stats}")


PRE: PASSED | {'bands': 4, 'height': 11838, 'width': 11838, 'dtype': 'float32', 'value_min': 0.0, 'value_max': 8725.53125, 'value_mean': 1203.2357177734375, 'value_std': 1256.840576171875, 'nan_pct': 0.0, 'zero_pct': 13.969542104437958}
POST: PASSED | {'bands': 4, 'height': 11828, 'width': 11828, 'dtype': 'float32', 'value_min': 0.0, 'value_max': 10669.9326171875, 'value_mean': 848.5025024414062, 'value_std': 1447.398193359375, 'nan_pct': 0.0, 'zero_pct': 63.89671256331165}


In [12]:
pre_scene  = pre_dl[0].path   if pre_dl[0].success  else None
post_scene = post_dl[0].path  if post_dl[0].success else None

if not pre_scene or not post_scene:
    raise RuntimeError(
        f"Download failed — pre: {pre_dl[0].error or 'ok'}, "
        f"post: {post_dl[0].error or 'ok'}"
    )

In [13]:
from pathlib import Path

# Extract real paths from the DownloadResult lists
pre_scene  = pre_dl[0].path  if pre_dl and pre_dl[0].success  else None
post_scene = post_dl[0].path if post_dl and post_dl[0].success else None

if not pre_scene or not post_scene:
    raise RuntimeError(
        f"Download failed — pre: {pre_dl[0].error if pre_dl else 'no result'}, "
        f"post: {post_dl[0].error if post_dl else 'no result'}"
    )

# Reproject via PyGeoVision's own rasterio-based reproject (avoids
# PyGeoFetch's native preprocessor.reproject() bugs seen earlier)
pre_scene_reproj  = client._pgf_bridge.preprocess_reproject(str(pre_scene),  crs='EPSG:4326')
post_scene_reproj = client._pgf_bridge.preprocess_reproject(str(post_scene), crs='EPSG:4326')

# Preprocess BOTH with identical parameters — critical for change detection accuracy
PRE_READY  = DATA_DIR / 'pre_preprocessed.tif'
POST_READY = DATA_DIR / 'post_preprocessed.tif'

def preprocess(scene_path, out_path, label):
    scl = next(iter(Path(scene_path).parent.rglob('*SCL*.tif')), None)
    r = client.prepare_for_ai(
        str(scene_path), stack_bands=BANDS, bbox=BBOX,
        scl_path=str(scl) if scl else None, normalise='scale_factor',
        scale_factor=10000.0, model_type='change_detection', output_path=str(out_path),
        use_pgf_preprocess=False,   # avoid PyGeoFetch's native clip/reproject bugs too
    )
    print(f"{label}: {r['steps']}  shape={r['shape']}  [{r['array'].min():.3f},{r['array'].max():.3f}]")
    return r

if pre_scene_reproj and post_scene_reproj:
    preprocess(pre_scene_reproj,  PRE_READY,  'PRE ')
    preprocess(post_scene_reproj, POST_READY, 'POST')
    print("Both scenes preprocessed and ready for ChangeFormer")

11:09:51 INFO [preprocessor] Reprojected → EPSG:4326 → results/nb04/pre/planetary_computer/S2A_MSIL2A_20210621T085601_R007_T36UUB_20210624T122020_repr_cog_reproj_EPSG_4326.tif


2026-07-31 11:09:51,901 [INFO] pygeofetch.processing.preprocessor: Reprojected → EPSG:4326 → results/nb04/pre/planetary_computer/S2A_MSIL2A_20210621T085601_R007_T36UUB_20210624T122020_repr_cog_reproj_EPSG_4326.tif


11:11:30 INFO [preprocessor] Reprojected → EPSG:4326 → results/nb04/post/planetary_computer/S2A_MSIL2A_20240628T090601_R050_T36UUA_20240628T131515_repr_cog_reproj_EPSG_4326.tif


2026-07-31 11:11:30,132 [INFO] pygeofetch.processing.preprocessor: Reprojected → EPSG:4326 → results/nb04/post/planetary_computer/S2A_MSIL2A_20240628T090601_R050_T36UUA_20240628T131515_repr_cog_reproj_EPSG_4326.tif
2026-07-31 11:11:55,360 [INFO] pygeovision.preprocess.core: Stacked 4 bands → /tmp/pgv_preprocess_vbnqt_l6/step_00_stacked.tif
2026-07-31 11:11:56,607 [INFO] pygeovision.preprocess.core: Clipped to bbox: 10980x10980 → 748x683 (0.4% of original) → /tmp/pgv_preprocess_vbnqt_l6/step_01_clipped.tif
2026-07-31 11:11:58,204 [INFO] pygeovision.preprocess.core: SCL mask: 4.6% pixels removed (classes [3, 7, 8, 9, 10, 11, 12, 14]) → /tmp/pgv_preprocess_vbnqt_l6/step_02_scl.tif
2026-07-31 11:11:58,444 [INFO] pygeovision.preprocess.core: Normalised (scale_factor) → /tmp/pgv_preprocess_vbnqt_l6/step_03_norm.tif
2026-07-31 11:11:59,051 [INFO] pygeovision.preprocess.core: Pipeline complete: results/nb04/pre_preprocessed.tif | shape=(4, 683, 748) | res=10.0m | steps=['stack(4 bands)', 'clip

PRE : ['stack(4 bands)', 'clip_bbox(30.48, 50.4, 30.58, 50.5)', 'scl(keep=[4, 5, 6])', 'normalise(scale_factor)', 'validate(change_detection)']  shape=(4, 683, 748)  [0.000,0.720]


2026-07-31 11:12:08,299 [INFO] pygeovision.preprocess.core: Stacked 4 bands → /tmp/pgv_preprocess_y5h7bgbn/step_00_stacked.tif
2026-07-31 11:12:08,468 [INFO] pygeovision.preprocess.core: Clipped to bbox: 10980x10980 → 748x1089 (0.7% of original) → /tmp/pgv_preprocess_y5h7bgbn/step_01_clipped.tif
2026-07-31 11:12:08,655 [INFO] pygeovision.preprocess.core: SCL mask: 64.3% pixels removed (classes [0, 1, 2, 3, 7, 8, 9, 10, 11, 12]) → /tmp/pgv_preprocess_y5h7bgbn/step_02_scl.tif
2026-07-31 11:12:08,797 [INFO] pygeovision.preprocess.core: Normalised (scale_factor) → /tmp/pgv_preprocess_y5h7bgbn/step_03_norm.tif
2026-07-31 11:12:09,017 [INFO] pygeovision.preprocess.core: Pipeline complete: results/nb04/post_preprocessed.tif | shape=(4, 1089, 748) | res=10.0m | steps=['stack(4 bands)', 'clip_bbox(30.48, 50.4, 30.58, 50.5)', 'scl(keep=[4, 5, 6])', 'normalise(scale_factor)']


POST: ['stack(4 bands)', 'clip_bbox(30.48, 50.4, 30.58, 50.5)', 'scl(keep=[4, 5, 6])', 'normalise(scale_factor)', 'validate(change_detection)']  shape=(4, 1089, 748)  [0.000,0.943]
Both scenes preprocessed and ready for ChangeFormer


In [14]:
CHANGE_MAP = DATA_DIR / 'change_map.tif'

cd = ChangeDetection(model_variant='changeformer', in_channels=len(BANDS))
cd.build()
res = cd.detect(str(PRE_READY), str(POST_READY), output_path=str(CHANGE_MAP))
print(f"Change coverage: {res['change_pct']:.2f}%")


RuntimeError: [enforce fail at alloc_cpu.cpp:127] err == 0. DefaultCPUAllocator: can't allocate memory: you tried to allocate 26119325232 bytes. Error code 12 (Cannot allocate memory)

In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())


In [8]:
SIEVED  = DATA_DIR / 'change_sieved.tif'
VECTOR  = DATA_DIR / 'change_polygons.geojson'
COG_OUT = DATA_DIR / 'change_cog.tif'
if CHANGE_MAP.exists():
    client.postprocess.sieve_filter(str(CHANGE_MAP), min_pixels=20, output_path=str(SIEVED))
    client.postprocess.vectorise(str(SIEVED), str(VECTOR), target_class=1,
        min_area_m2=100.0, simplify_tolerance=1.0)
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    with open(VECTOR) as f: gj = json.load(f)
    print(f"Changed polygons: {len(gj['features'])}")
    print(f"Changed area    : {stats.get(1,{}).get('area_ha',0):.1f} ha")


**NB04 — Change Detection** complete.
- Study area: Kyiv Ukraine
- Sensor: Sentinel-2 4-band
- Model: ChangeFormer MiT-B2
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG